# Treinamento Wake Word Letícia (microWakeWord)

Este notebook foi desenvolvido especificamente para treinar palavras de ativação destinadas a dispositivos **ESPHome (como o HA Voice PE)** utilizando o motor `microWakeWord`.

## Vantagens desta versão (v01 - Inception):
- **Foco no Hardware:** Gera o ficheiro `.tflite` quantizado necessário para rodar nativamente nos chips ESP32-S3.
- **Velocidade & Estabilidade:** Baixa os datasets de ruído negativo já pré-processados (`.mmap`) do repositório oficial, evitando horas de processamento e crashes por falta de memória.
- **Arquitetura Inception:** Usa a arquitetura padrão do microWakeWord, garantindo o máximo de compatibilidade e contornando os *bugs* de parsing da versão `mixednet`.
- **Correção de Áudio Integrada:** Converte automaticamente os áudios gerados pelo Piper (22.050Hz) para os rigorosos **16.000Hz** exigidos pelo motor de treino.

---

### Etapa 1: Setup do Ambiente e Dependências

Ative a GPU T4 no Colab (`Ambiente de execução > Alterar tipo de ambiente de execução`) antes de começar. Depois, execute esta célula.

In [ ]:
import os
import sys
import subprocess
import shutil

print("=" * 60)
print(" ETAPA 1: Setup do Ambiente microWakeWord")
print("=" * 60)

def run_cmd(cmd, desc):
    print(f"\n[Executando] {desc}...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f" ✅ Sucesso!")
    else:
        print(f" ❌ Erro:\n{result.stderr[-500:]}")
        raise RuntimeError(f"Falha em: {desc}")

run_cmd("sudo apt-get update && sudo apt-get install -y libespeak-ng-dev", "Dependências do sistema (espeak-ng)")
run_cmd(f"{sys.executable} -m pip install -q git+https://github.com/kahrendt/microWakeWord.git", "Instalação do microWakeWord")
run_cmd(f"{sys.executable} -m pip install -q piper-tts piper-phonemize-cross==1.2.1", "Instalação do Piper TTS")

print("\n[Verificando Instalações]")
try:
    import microwakeword
    import torch
    import torchaudio
    print(f" ✅ microWakeWord carregado com sucesso.")
    print(f" ✅ PyTorch versão: {torch.__version__}")
    
    if torch.cuda.is_available():
        print(f" ✅ GPU Ativa: {torch.cuda.get_device_name(0)}")
    else:
        print(f" ⚠️ ALERTA: Nenhuma GPU detectada. Mude o ambiente de execução para T4 GPU!")
        
    if shutil.which("piper"):
        print(f" ✅ Piper CLI detectado em: {shutil.which('piper')}")
    else:
        print(f" ❌ Erro: Piper CLI não encontrado.")
        
except ImportError as e:
    print(f" ❌ Erro ao importar bibliotecas: {e}")

print("\n" + "=" * 60)
print(" ETAPA 1 CONCLUÍDA!")
print("=" * 60)

### Etapa 2: Gerar Amostras de Áudio com Piper

Vamos usar a voz `pt_BR-faber-medium` e gerar 1000 amostras com variações de velocidade e ruído de fundo, convertendo-as de seguida rigorosamente para 16kHz.

In [ ]:
import uuid
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

print("=" * 60)
print(" ETAPA 2: Geração de Amostras TTS e Conversão (16kHz)")
print("=" * 60)

TARGET_WORD = "letícia"
N_SAMPLES = 1000
SAMPLES_DIR = "positive_samples"
N_WORKERS = 4

os.makedirs("piper_voices_ptbr", exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

HF_BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main"
voice_name = "pt_BR-faber-medium"
voice_path = f"piper_voices_ptbr/{voice_name}.onnx"

if not os.path.exists(voice_path):
    print(f"[Baixando Voz Piper: {voice_name}]")
    subprocess.run(["wget", "-q", f"{HF_BASE}/pt/pt_BR/faber/medium/{voice_name}.onnx", "-O", voice_path], check=True)
    subprocess.run(["wget", "-q", f"{HF_BASE}/pt/pt_BR/faber/medium/{voice_name}.onnx.json", "-O", f"{voice_path}.json"], check=True)
    print(" ✅ Download concluído.")

LENGTH_SCALES = [0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25]
NOISE_SCALES  = [0.50, 0.60, 0.667, 0.70, 0.80, 0.90, 0.98]
NOISE_WS      = [0.50, 0.60, 0.70, 0.80, 0.90, 0.98]

def gen_one_clip(args):
    word, voice, out_path = args
    try:
        r = subprocess.run(
            ["piper", "--model", voice, "--output_file", out_path,
             "--length-scale", str(random.choice(LENGTH_SCALES)),
             "--noise-scale",  str(random.choice(NOISE_SCALES)),
             "--noise-w",      str(random.choice(NOISE_WS))],
            input=word, capture_output=True, text=True, timeout=30
        )
        return os.path.exists(out_path) and os.path.getsize(out_path) > 0
    except:
        return False

existing = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]
needed = max(0, N_SAMPLES - len(existing))

if needed > 0:
    print(f"\n[Gerando {needed} amostras para '{TARGET_WORD}']")
    tasks = [(TARGET_WORD, voice_path, os.path.join(SAMPLES_DIR, f"{uuid.uuid4().hex}.wav")) for _ in range(needed)]
    count = 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        for future in as_completed([executor.submit(gen_one_clip, t) for t in tasks]):
            if future.result(): count += 1
            if count % 100 == 0: print(f"  ... gerados {count}/{needed} clips")
    print(f" ✅ {count} amostras geradas em {(time.time() - t0)/60:.1f} min")
else:
    print(f"\n ✅ {len(existing)} amostras já existem.")

print("\n[Verificando/Convertendo Sample Rate para 16kHz]")
wavs = [f for f in os.listdir(SAMPLES_DIR) if f.endswith(".wav")]
bad = []
for f in wavs:
    try:
        info = torchaudio.info(f"{SAMPLES_DIR}/{f}")
        if info.sample_rate != 16000: bad.append((f, info.sample_rate))
    except: pass

if bad:
    print(f" ⚠️ Encontrados {len(bad)} clips fora dos 16kHz. Convertendo...")
    for fname, sr in bad:
        path = f"{SAMPLES_DIR}/{fname}"
        wf, _ = torchaudio.load(path)
        if wf.shape[0] > 1: wf = wf.mean(dim=0, keepdim=True)
        wf = torchaudio.transforms.Resample(sr, 16000)(wf)
        torchaudio.save(path, wf, 16000)
    print(" ✅ Conversão concluída.")
else:
    print(" ✅ Todos os áudios estão corretos (16kHz).")

print("\n" + "=" * 60)
print(" ETAPA 2 CONCLUÍDA!")
print("=" * 60)

### Etapa 3: Download e Preparação dos Datasets Negativos

O microWakeWord precisa de saber distinguir o que *não* é a palavra Letícia. Para isso, precisamos do ruído de fundo (conversas, barulhos). Como gerar estas *features* (`.mmap`) demora muito tempo, vamos descarregar os ficheiros que o autor oficial já compilou.

In [ ]:
print("=" * 60)
print(" ETAPA 3: Download Datasets Negativos Pré-processados (.mmap)")
print("=" * 60)

base_dir = os.path.abspath(os.getcwd())
mmap_base_url = "https://github.com/kahrendt/microWakeWord/releases/download/v1.0.0"

mmap_datasets = [
    ("speech.mmap", "negative_datasets_mmap/speech"),
    ("dinner_party.mmap", "negative_datasets_mmap/dinner_party"),
    ("no_speech.mmap", "negative_datasets_mmap/no_speech"),
    ("dinner_party_eval.mmap", "negative_datasets_mmap/dinner_party_eval"),
]

for filename, dest_dir in mmap_datasets:
    dest_path = os.path.join(base_dir, dest_dir)
    os.makedirs(dest_path, exist_ok=True)
    file_path = os.path.join(dest_path, filename)
    
    if not os.path.exists(file_path):
        print(f" [Baixando] {filename}...")
        subprocess.run(["wget", "-q", "--show-progress", "-c", f"{mmap_base_url}/{filename}", "-O", file_path], check=True)
        print(f" ✅ {filename} baixado.")
    else:
        print(f" ✅ {filename} já existe.")

print("\n" + "=" * 60)
print(" ETAPA 3 CONCLUÍDA!")
print("=" * 60)

### Etapa 4: Configuração do YAML e Geração das Features Positivas

Cria o ficheiro `training_parameters.yaml` ligando todas as pastas, usando caminhos absolutos para evitar erros de leitura e executa a conversão dos áudios da Letícia para o formato `.mmap`.

In [ ]:
import yaml
import glob

print("=" * 60)
print(" ETAPA 4: Configuração YAML e Feature Generation")
print("=" * 60)

config = {
    "window_step_ms": 10,
    "train_dir": os.path.join(base_dir, "trained_models/wakeword"),
    "features": [
        {
            "features_dir": os.path.join(base_dir, "generated_augmented_features"),
            "sampling_weight": 2.0,
            "penalty_weight": 1.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },
        {
            "features_dir": os.path.join(base_dir, "negative_datasets_mmap/speech"),
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": os.path.join(base_dir, "negative_datasets_mmap/dinner_party"),
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": os.path.join(base_dir, "negative_datasets_mmap/no_speech"),
            "sampling_weight": 5.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": os.path.join(base_dir, "negative_datasets_mmap/dinner_party_eval"),
            "sampling_weight": 0.0,
            "penalty_weight": 0.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
    ],
    "training_steps": [50000],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates": [0.001],
    "batch_size": 128,
    "time_mask_max_size": [0], "time_mask_count": [0],
    "freq_mask_max_size": [0], "freq_mask_count": [0],
    "eval_step_interval": 1000,
    "clip_duration_ms": 1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall"
}

yaml_path = os.path.join(base_dir, "training_parameters.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
print(" ✅ training_parameters.yaml gerado com sucesso.")

print("\n[Gerando arquivo .mmap das amostras positivas]")
positive_out = os.path.join(base_dir, "generated_augmented_features")
if not os.path.exists(positive_out) or len(glob.glob(f"{positive_out}/**/*.mmap", recursive=True)) == 0:
    os.makedirs(positive_out, exist_ok=True)
    cmd = [
        sys.executable, "-m", "microwakeword.data",
        "--wav_dir", os.path.join(base_dir, "positive_samples"),
        "--output_dir", positive_out,
        "--config", yaml_path
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f" ❌ Falha:\n{r.stderr[:500]}")
        raise RuntimeError("Geração de features falhou.")
    print(" ✅ Features positivas processadas.")
else:
    print(" ✅ Features positivas prontas.")

print("\n" + "=" * 60)
print(" ETAPA 4 CONCLUÍDA!")
print("=" * 60)

### Etapa 5: Treinamento Neural (O Grande Momento)

Agora vamos juntar todos os dados e treinar o modelo, utilizando a arquitetura `inception` por ser altamente otimizada e livre dos bugs de parsing de outras arquiteturas. **Este processo demora.**

In [ ]:
print("=" * 60)
print(" ETAPA 5: Treinamento Neural (microWakeWord Inception)")
print("=" * 60)

print(" 🚀 Iniciando Treinamento... (Pode demorar entre 1 a 2 horas)")

train_cmd = [
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config", yaml_path,
    "--train", "1",
    "--restore_checkpoint", "1",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "--use_weights", "best_weights",
    "inception" 
]

try:
    process = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
    process.stdout.close()
    process.wait()
    if process.returncode != 0:
        print(f"\n❌ ERRO DETECTADO: Falhou com código {process.returncode}.")
        raise RuntimeError("O Treinamento foi abortado.")
except Exception as e:
    print(f"\n❌ Exceção Crítica de execução: {e}")

print("\n" + "=" * 60)
print(" ETAPA 5 CONCLUÍDA!")
print("=" * 60)

### Etapa 6: Exportação, Manifesto e Download

Vamos recolher o ficheiro `.tflite` gerado, criar o Manifesto JSON necessário para injetar no ESPHome do seu Home Assistant Voice PE, e efetuar o download para o seu computador.

In [ ]:
import json
from google.colab import files

print("=" * 60)
print(" ETAPA 6: Empacotamento para ESPHome e Download")
print("=" * 60)

TFLITE_PATH = os.path.join(base_dir, "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")

if not os.path.exists(TFLITE_PATH):
    candidates = glob.glob(os.path.join(base_dir, "trained_models/**/*.tflite"), recursive=True)
    if candidates:
        TFLITE_PATH = sorted(candidates, key=os.path.getmtime)[-1]

if os.path.exists(TFLITE_PATH):
    size_kb = os.path.getsize(TFLITE_PATH) / 1024
    print(f" ✅ Modelo TFLite localizado ({size_kb:.1f} KB):\n    {TFLITE_PATH}")
else:
    raise FileNotFoundError("❌ Modelo .tflite não foi encontrado. O Treinamento não gerou o ficheiro final.")

os.makedirs("output", exist_ok=True)
tflite_dest = "output/leticia_mww.tflite"
shutil.copy(TFLITE_PATH, tflite_dest)

# ── CRIAÇÃO DO MANIFESTO JSON (ESPHome) ──
# NOTA IMPORTANTE: Após descarregar os ficheiros, deverá colocar o ficheiro leticia_mww.tflite num repositório 
# alojado publicamente (como o GitHub Releases) e colocar a hiperligação no campo 'model' abaixo antes de colocar no ESPHome.

manifest = {
    "type": "micro",
    "wake_word": "Leticia",
    "author": "Maycon Willian",
    "website": "https://github.com/visaodeempresa/ha-wakeword-leticia",
    "model": "https://github.com/visaodeempresa/ha-wakeword-leticia/releases/download/v1.0-mww/leticia_mww.tflite",
    "trained_languages": ["pt"],
    "version": 1,
    "micro": {
        "probability_cutoff": 0.5,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 26080,
        "minimum_esphome_version": "2024.7.0",
    },
}

json_dest = "output/leticia_mww.json"
with open(json_dest, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(" ✅ Ficheiro leticia_mww.json criado com sucesso.")

print("\n[A Preparar o Download Automático]")
print(" ⚠️ O seu navegador poderá pedir permissão para descarregar múltiplos ficheiros. Aceite.")
try:
    files.download(tflite_dest)
    files.download(json_dest)
except Exception as e:
    print(f" ⚠️ Não foi possível iniciar o download automático: {e}")
    print(" Por favor, faça o download manualmente navegando pela aba esquerda de Ficheiros até à pasta 'output'.")

print("\n" + "=" * 60)
print(" PROCESSO COMPLETAMENTE CONCLUÍDO!")
print("=" * 60)
print("Próximos Passos no Home Assistant ESPHome:")
print("1. Alojar leticia_mww.tflite online.")
print("2. Editar leticia_mww.json para apontar para a URL do ficheiro alojado.")
print("3. Alojar o leticia_mww.json online.")
print("4. No ESPHome (via Home Assistant), editar o dispositivo HA Voice PE e adicionar a URL do JSON em 'models' na secção 'micro_wake_word'.")
print("5. Re-instalar o Firmware via OTA.")